In [ ]:
# TWO-STAGE PLANT DISEASE CLASSIFIER

# SETUP
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

dataset_dir = '/content/plant_disease_dataset'
zip_path = '/content/plant_disease_dataset.zip'

if not os.path.exists(dataset_dir):
    if not os.path.exists(zip_path):
        print("📦 Downloading dataset from Kaggle...")
        !kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content --force
    else:
        print("✅ Dataset zip file already exists, skipping download.")

    print("🗂️ Extracting dataset...")
    !unzip -q /content/new-plant-diseases-dataset.zip -d {dataset_dir}
    print("✅ Extraction complete!")
else:
    print("✅ Dataset already extracted, skipping download and unzip.")

dataset_path = '/content/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)'
train_dir = os.path.join(dataset_path, 'train')
val_dir = os.path.join(dataset_path, 'valid')

# MODEL 1 (PLANT ONLY)

def create_plant_only_structure(src_dir, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    for cls in os.listdir(src_dir):
        if '___' in cls:
            plant = cls.split('___')[0]
            src_cls_path = os.path.join(src_dir, cls)
            dst_cls_path = os.path.join(dest_dir, plant)
            os.makedirs(dst_cls_path, exist_ok=True)
            for img_file in os.listdir(src_cls_path):
                shutil.copy(os.path.join(src_cls_path, img_file),
                            os.path.join(dst_cls_path, img_file))

create_plant_only_structure(train_dir, '/content/plant_only_train')
create_plant_only_structure(val_dir, '/content/plant_only_valid')

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# image generators to prevent overfitting

train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator_plant = train_datagen.flow_from_directory(
    '/content/plant_only_train',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_generator_plant = val_datagen.flow_from_directory(
    '/content/plant_only_valid',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# define plant model

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import tensorflow as tf

tf.keras.backend.clear_session()

model_plant = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(train_generator_plant.num_classes, activation='softmax')
])

model_plant.compile(optimizer=Adam(1e-3),
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model_plant.summary()

# train plant model

history_plant = model_plant.fit(
    train_generator_plant,
    steps_per_epoch=train_generator_plant.samples // train_generator_plant.batch_size,
    validation_data=val_generator_plant,
    validation_steps=val_generator_plant.samples // val_generator_plant.batch_size,
    epochs=15,
    callbacks=[lr_scheduler, early_stop]
)

model_plant.save('/content/plant_classifier.h5')
print("✅ Plant classifier saved as plant_classifier.h5")

# DISEASE CLASSIFIERS (PER PLANT)

plants = sorted(os.listdir(train_dir))
disease_models = {}

for plant_name in set([p.split('___')[0] for p in plants]):
    print(f"\n🌿 Training disease classifier for {plant_name}...")

    disease_train_dir = f'/content/{plant_name}_train'
    disease_val_dir = f'/content/{plant_name}_valid'
    os.makedirs(disease_train_dir, exist_ok=True)
    os.makedirs(disease_val_dir, exist_ok=True)

    for cls in os.listdir(train_dir):
        if cls.startswith(plant_name + '___'):
            shutil.copytree(os.path.join(train_dir, cls),
                            os.path.join(disease_train_dir, cls.replace(plant_name + '___', '')),
                            dirs_exist_ok=True)
    for cls in os.listdir(val_dir):
        if cls.startswith(plant_name + '___'):
            shutil.copytree(os.path.join(val_dir, cls),
                            os.path.join(disease_val_dir, cls.replace(plant_name + '___', '')),
                            dirs_exist_ok=True)

    # image generators to prevent overfitting

    train_gen = train_datagen.flow_from_directory(
        disease_train_dir,
        target_size=(128, 128),
        batch_size=32,
        class_mode='categorical',
        shuffle=True
    )
    val_gen = val_datagen.flow_from_directory(
        disease_val_dir,
        target_size=(128, 128),
        batch_size=32,
        class_mode='categorical',
        shuffle=False
    )

    # define disease model (same structure)
    model_disease = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(train_gen.num_classes, activation='softmax')
    ])

    model_disease.compile(optimizer=Adam(1e-3),
                          loss='categorical_crossentropy',
                          metrics=['accuracy'])

    # train
    history_disease = model_disease.fit(
        train_gen,
        steps_per_epoch=train_gen.samples // train_gen.batch_size,
        validation_data=val_gen,
        validation_steps=val_gen.samples // val_gen.batch_size,
        epochs=15,
        callbacks=[lr_scheduler, early_stop],
        verbose=1
    )

    model_disease.save(f'/content/{plant_name}_disease_classifier.h5')
    print(f"✅ Saved {plant_name}_disease_classifier.h5")

# DOWNLOAD TRAINED MODELS

from google.colab import files

files.download('/content/plant_classifier.h5')

for plant_name in set([p.split('___')[0] for p in plants]):
    files.download(f'/content/{plant_name}_disease_classifier.h5')


In [ ]:
# VERIFY DOWNLOADS

plants = sorted([
    d for d in os.listdir('/content')
    if d.endswith('_train') and os.path.isdir(os.path.join('/content', d))
])

for folder in plants:
    plant_name = folder.replace('_train', '')
    disease_class_names = sorted([
        d for d in os.listdir(f"/content/{folder}")
        if os.path.isdir(os.path.join(f"/content/{folder}", d))
    ])
    print(f"{plant_name}_classes = {disease_class_names}\n")
